In [13]:
from pyspark.sql import SparkSession
from pyspark.ml.fpm import FPGrowth
import pyspark.sql.functions as F

# Initialize local Spark
spark = (
    SparkSession.builder.appName("Pediatric_ADR_Mining_Local")
    .config("spark.driver.memory", "18g")
    .config("spark.executor.memory", "4g")
    .config("spark.driver.maxResultSize", "4g")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .getOrCreate()
)

In [22]:
fp = 'arm_transactions_2026052411.parquet'
df = spark.read.parquet(f"./{fp}")

In [23]:
df.count()

44788

In [25]:
# FP-Growth at the "Signal Detection" threshold
fpGrowth = FPGrowth(
    itemsCol="adr_case",
    predictionCol="recommendations_",
    minSupport=0.0005,
    minConfidence=0.7,  # Lower confidence to catch more associations
)

model = fpGrowth.fit(df)

In [26]:
# Get the rules
rules = model.associationRules

In [27]:
# IMPORTANT: Filter for actual insights (Consequents should be Reactions)
# This removes the "noise" of demographic-to-demographic rules
interesting_rules = rules.filter(
    F.array_join("consequent", ",").contains("reaction_")
    | F.array_join("consequent", ",").contains("outc_")
).sort(F.desc("lift"))

In [28]:
interesting_rules.show(20, truncate=False)

+-------------------------------------------------------------------------------------------------------------------------------+-----------------------------+------------------+------------------+--------------------+
|antecedent                                                                                                                     |consequent                   |confidence        |lift              |support             |
+-------------------------------------------------------------------------------------------------------------------------------+-----------------------------+------------------+------------------+--------------------+
|[reaction_metabolic_acidosis, drug_metformin, outc_ho, demo_age_bin_teenager, demo_sex_f, demo_origin_us]                      |[reaction_hyperlactacidaemia]|0.9583333333333334|1788.4097222222224|5.135304099312315E-4|
|[reaction_metabolic_acidosis, drug_metformin, outc_ho, demo_weight_40-60 kg, demo_age_bin_teenager]                        

In [29]:
interesting_rules.count()

17376

In [30]:
interesting_rules.write.mode("overwrite").format("parquet").save("mined_rules/arm_mined_2026052411.parquet")

In [31]:
spark.stop()